# MPIA Arxiv on Deck 2

Contains the steps to produce the paper extractions.

In [1]:
# Imports
import os
from IPython.display import Markdown, display
from tqdm.notebook import tqdm
import warnings
from PIL import Image 
import re

# requires arxiv_on_deck_2

from arxiv_on_deck_2.arxiv2 import (get_new_papers, 
                                    get_paper_from_identifier,
                                    retrieve_document_source, 
                                    get_markdown_badge)
from arxiv_on_deck_2 import (latex,
                             latex_bib,
                             mpia,
                             highlight_authors_in_list)

# Sometimes images are really big
Image.MAX_IMAGE_PIXELS = 1000000000 

In [2]:
# Some useful definitions.

class AffiliationWarning(UserWarning):
    pass

class AffiliationError(RuntimeError):
    pass

def validation(source: str):
    """Raises error paper during parsing of source file
    
    Allows checks before parsing TeX code.
    
    Raises AffiliationWarning
    """
    check = mpia.affiliation_verifications(source, verbose=True)
    if check is not True:
        raise AffiliationError("mpia.affiliation_verifications: " + check)

        
warnings.simplefilter('always', AffiliationWarning)


def get_markdown_qrcode(paper_id: str):
    """ Generate a qrcode to the arxiv page using qrserver.com
    
    :param paper: Arxiv paper
    :returns: markdown text
    """
    url = r"https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="
    txt = f"""<img src={url}"https://arxiv.org/abs/{paper_id}">"""
    txt = '<div id="qrcode">' + txt + '</div>'
    return txt


def clean_non_western_encoded_characters_commands(text: str) -> str:
    """ Remove non-western encoded characters from a string
    List may need to grow.
    
    :param text: the text to clean
    :return: the cleaned text
    """
    text = re.sub(r"(\\begin{CJK}{UTF8}{gbsn})(.*?)(\\end{CJK})", r"\2", text)
    return text


def get_initials(name: str) -> str:
    """ Get the short name, e.g., A.-B. FamName
    :param name: full name
    :returns: initials
    """
    initials = []
    # account for non western names often in ()
    if '(' in name:
        name = clean_non_western_encoded_characters_commands(name)
        suffix = re.findall(r"\((.*?)\)", name)[0]
        name = name.replace(f"({suffix})", '')
    else:
        suffix = ''
    split = name.split()
    for token in split[:-1]:
        if '-' in token:
            current = '-'.join([k[0] + '.' for k in token.split('-')])
        else:
            current = token[0] + '.'
        initials.append(current)
    initials.append(split[-1].strip())
    if suffix:
        initials.append(f"({suffix})")
    return ' '.join(initials)

## get list of arxiv paper candidates

We use the MPIA mitarbeiter list webpage from mpia.de to get author names
We then get all new papers from Arxiv and match authors

In [3]:
# deal with the author list and edge cases of people that cannot be consistent on their name  

def filter_non_scientists(name: str) -> bool:
    """ Loose filter on expected authorships

    removing IT, administration, technical staff
    :param name: name
    :returns: False if name is not a scientist
    """
    remove_list = ['Licht', 'Binroth', 'Witzel', 'Jordan',
                   'Zähringer', 'Scheerer', 'Hoffmann', 'Düe',
                   'Hellmich', 'Enkler-Scharpegge', 'Witte-Nguy',
                   'Dehen', 'Beckmann', 'Jager', 'Jäger'
                  ]

    for k in remove_list:
        if k in name:
            return False
    return True

def add_author_to_list(author_list: list) -> list:
    """ Add author to list if not already in list
    
    :param author: author name
    :param author_list: list of authors
    :returns: updated list of authors
    """
    add_list = ['T. Henning', 'R. E. Hviding']

    for author in add_list:
        if author not in author_list:
            author_list.append(author)
    return author_list

# get list from MPIA website
# filter for non-scientists (mpia.get_mpia_mitarbeiter_list() does some filtering)
mpia_authors = [k[1] for k in mpia.get_mpia_mitarbeiter_list() if filter_non_scientists(k[1])]
# add some missing author because of inconsistencies in their MPIA name and author name on papers
mpia_authors = add_author_to_list(mpia_authors)

In [4]:
new_papers = get_new_papers()
# add manual references
add_paper_refs = []
new_papers.extend([get_paper_from_identifier(k) for k in add_paper_refs])

def robust_call(fn, value, *args, **kwargs):
    try:
        return fn(value, *args, **kwargs)
    except Exception:
        return value

candidates = []
for paperk in new_papers:
    # Check author list with their initials
    normed_author_list = [robust_call(mpia.get_initials, k) for k in paperk['authors']]
    hl_authors = highlight_authors_in_list(normed_author_list, mpia_authors, verbose=True)
    matches = [(hl, orig) for hl, orig in zip(hl_authors, paperk['authors']) if 'mark' in hl]
    paperk['authors'] = hl_authors
    if matches:
        # only select paper if an author matched our list
        candidates.append(paperk)
print("""Arxiv has {0:,d} new papers today""".format(len(new_papers)))        
print("""          {0:,d} with possible author matches""".format(len(candidates)))

A. d. Graaff  ->  A. D. Graaff  |  ['A. D. Graaff']
I. J. M. Crossfield  ->  I. J. M. Crossfield  |  ['I. J. M. Crossfield']
S. Kraus  ->  S. Kraus  |  ['S. Kraus']
J. Li  ->  J. Li  |  ['J. Li']
M. Zhang  ->  M. Zhang  |  ['M. Zhang']
M. Zhang  ->  M. Zhang  |  ['M. Zhang']
R. Hoppe  ->  R. Hoppe  |  ['R. Hoppe']
M. Bergemann  ->  M. Bergemann  |  ['M. Bergemann']
P. Eitner  ->  P. Eitner  |  ['P. Eitner']
N. Storm  ->  N. Storm  |  ['N. Storm']
M. Zhang  ->  M. Zhang  |  ['M. Zhang']
J. G. Lobos  ->  J. G. Lobos  |  ['J. G. Lobos']
E. Schinnerer  ->  E. Schinnerer  |  ['E. Schinnerer']
Arxiv has 71 new papers today
          9 with possible author matches


# Parse sources and generate relevant outputs

From the candidates, we do the following steps:
* get their tarball from ArXiv (and extract data)
* find the main .tex file: find one with \documentclass{...} (sometimes it's non trivial)
* Check affiliations with :func:`validation`, which uses :func:`mpia.affiliation_verifications`
* If passing the affiliations: we parse the .tex source
   * inject sub-documents into the main (flatten the main document)
   * parse structure, extract information (title, abstract, authors, figures...)
   * handles `\graphicspath` if provided
* Generate the .md document.

In [5]:
documents = []
failed = []
for paper in tqdm(candidates):
    # debug crap
    paper['identifier'] = paper['identifier'].lower().replace('arxiv:', '').replace(r'\n', '').strip()
    paper_id = paper['identifier']
    
    folder = f'tmp_{paper_id}'

    try:
        if not os.path.isdir(folder):
            folder = retrieve_document_source(f"{paper_id}", f'tmp_{paper_id}')
        
        try:
            doc = latex.LatexDocument(folder, validation=validation)    
        except AffiliationError as affilerror:
            msg = f"ArXiv:{paper_id:s} is not an MPIA paper... " + str(affilerror)
            failed.append((paper, "affiliation error: " + str(affilerror) ))
            continue
        
        # Hack because sometimes author parsing does not work well
        if (len(doc.authors) != len(paper['authors'])):
            doc._authors = paper['authors']
        else:
            # highlight authors (FIXME: doc.highlight_authors)
            # done on arxiv paper already
            doc._authors = highlight_authors_in_list(
                [get_initials(k) for k in doc.authors], 
                mpia_authors, verbose=True)
        if (doc.abstract) in (None, ''):
            doc._abstract = paper['abstract']
            
        doc.comment = (get_markdown_badge(paper_id) + 
                       "<mark>Appeared on: " + paper['date'] + "</mark> - ")
        if paper['comments']:
            doc.comment += " _" + paper['comments'] + "_"
        
        full_md = doc.generate_markdown_text()
        
        full_md += get_markdown_qrcode(paper_id)
        
        # replace citations
        try:
            bibdata = latex_bib.LatexBib.from_doc(doc)
            full_md = latex_bib.replace_citations(full_md, bibdata)
        except Exception as e:
            print("Issues with the citations")
            print(e)
        
        documents.append((paper_id, full_md))
    except Exception as e:
        warnings.warn(latex.LatexWarning(f"{paper_id:s} did not run properly\n" +
                                         str(e)
                                        ))
        failed.append((paper, "latex error " + str(e)))

  0%|          | 0/9 [00:00<?, ?it/s]

Retrieving document from  https://arxiv.org/e-print/2611.08612
extracting tarball to tmp_2611.08612...

 done.


A. d. Graaff  ->  A. D. Graaff  |  ['A. D. Graaff']


Issues with the citations
undefined string in line 2759: june
Retrieving document from  https://arxiv.org/e-print/2611.08676
extracting tarball to tmp_2611.08676...

 done.
Retrieving document from  https://arxiv.org/e-print/2611.08824
extracting tarball to tmp_2611.08824...

... (5 outputs left out)

 done.
Retrieving document from  https://arxiv.org/e-print/2611.09370
extracting tarball to tmp_2611.09370... done.


Found 161 bibliographic references in tmp_2611.09370/openmulti3d.bbl.
Issues with the citations
syntax error in line 942: '=' expected
Retrieving document from  https://arxiv.org/e-print/2611.09743
extracting tarball to tmp_2611.09743... done.
Retrieving document from  https://arxiv.org/e-print/2611.09875


extracting tarball to tmp_2611.09875... done.


Issues with the citations
plugin pybtex.database.input.suffixes for suffix .tex not found


### Export the logs

Throughout, we also keep track of the logs per paper. see `logs-{today date}.md` 

In [6]:
import datetime
today = str(datetime.date.today())
logfile = f"_build/html/logs/log-{today}.md"


with open(logfile, 'w') as logs:
    # Success
    logs.write(f'# Arxiv on Deck 2: Logs - {today}\n\n')
    logs.write("""* Arxiv had {0:,d} new papers\n""".format(len(new_papers)))
    logs.write("""    * {0:,d} with possible author matches\n\n""".format(len(candidates)))
    logs.write("## Sucessful papers\n\n")
    display(Markdown("## Successful papers"))
    success = [k[0] for k in documents]
    for candid in candidates:
        if candid['identifier'].split(':')[-1] in success:
            display(candid)
            logs.write(candid.generate_markdown_text() + '\n\n')

    ## failed
    logs.write("## Failed papers\n\n")
    display(Markdown("## Failed papers"))
    failed = sorted(failed, key=lambda x: x[1])
    current_reason = ""
    for paper, reason in failed:
        if 'affiliation' in reason:
            color = 'green'
        else:
            color = 'red'
        data = Markdown(
                paper.generate_markdown_text() + 
                f'\n|<p style="color:{color:s}"> **ERROR** </p>| <p style="color:{color:s}">{reason:s}</p> |'
               )
        if reason != current_reason:
            logs.write(f'### {reason:s} \n\n')
            current_reason = reason
        logs.write(data.data + '\n\n')
        
        # only display here the important errors (all in logs)
        # if color in ('red',):
        display(data)

## Successful papers


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.08612-b31b1b.svg)](https://arxiv.org/abs/2611.08612) | **Jeans Dynamical Constraints on the IMF in Massive Quiescent Galaxies at $z\sim0.7$ from JWST/NIRSpec**  |
|| M. Slob, et al. -- incl., <mark>A. d. Graaff</mark> |
|*Appeared on*| *2026-11-09*|
|*Comments*| *Resubmitted to A&A after minor revisions; 15 pages, 6 figures (excluding appendices)*|
|**Abstract**|            We present dynamical masses for nine massive quiescent galaxies at $z\sim0.7$ from the JWST-IMFERNO program, measured from ultra-deep ($\sim32$hr), spatially resolved NIRSpec/MSA spectroscopy combined with NIRCam imaging. The most massive of these galaxies have previously been found to have a bottom-heavy initial mass function (IMF), based on measurements of dwarf-sensitive absorption features. Here, we examine whether the stellar masses inferred for these IMFs are consistent with the dynamical masses of these galaxies. We derive the dynamical masses by including Jeans Anisotropic Modelling within a forward-modelling framework that accounts for instrumental optics, source morphology, and geometric effects. We compare the resulting dynamical masses to the stellar masses for different IMF assumptions. A Milky Way IMF produces physically plausible dynamical-to-stellar mass ratios but implies central dark matter fractions ($f_{\rm DM,1R_e}=0.54\pm0.06$) that are a factor $\sim2$ higher compared to local ETGs and simulations. When uniformly applying the IMF that was measured from absorption features, the mean dark matter fraction becomes low ($f_{\rm DM,1R_e}=0.02\pm0.18$), and the stellar masses exceed the dynamical mass at $2\sigma$ significance for two out of seven galaxies in the sample. When we adopt the measured IMF only in the central $R_e$ and use a MW IMF in the outskirts, we find realistic dark matter fractions ($f_{\rm DM,1R_e}=0.28\pm0.12$) and consistent dynamical masses. These results support a scenario where the cores of the most massive quiescent galaxies at $z\sim0.7$ had a bottom-heavy IMF, similar to those observed in the centres of local early-type galaxies. Our results are consistent with an inside-out growth scenario, in which size and mass growth of massive galaxies proceeds after quenching through minor mergers, leaving the bottom-heavy cores mostly intact.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.08983-b31b1b.svg)](https://arxiv.org/abs/2611.08983) | **Identifying Chemically Coherent Stellar Families in the Galactic High-α Disk with Low-Resolution Spectroscopy**  |
|| T. Wu, et al. -- incl., <mark>M. Zhang</mark> |
|*Appeared on*| *2026-11-09*|
|*Comments*| *21 pages, 14 figures, accepted to ApJ*|
|**Abstract**|            Identification of chemical substructures in the Galactic high-{\alpha} disk provides valuable insight into its formation history. In this work, we demonstrate the capability of low-resolution spectroscopy for this purpose by applying DBSCAN clustering to 87,275 red giants in the high-{\alpha} disk using 18 elemental abundances derived from LAMOST low-resolution spectra (R~1800). We assess the effectiveness of this approach by comparing the clustering results from the low-resolution data with those obtained by applying the same procedure to high-resolution survey abundances and to mock catalogs. These comparisons suggest that, despite the relatively large uncertainties in individual elemental abundances derived from low-resolution spectra, the astrophysical coherence among different elements provides sufficient information to identify chemically coherent families that may share similar enrichment histories without necessarily sharing a common birth environment. After scanning the parameter space of the DBSCAN procedure and merging the results from different parameter combinations, we identify 133 candidate chemical families containing 4,642 unique stars (5.3% of the high-{\alpha} disk sample). These families exhibit diverse multi-element abundance patterns, with particularly prominent differences in neutron-capture elements at nearly identical [Fe/H] values. These families therefore offer new constraints on the diversity of chemical enrichment pathways and on the assembly history of the high-{\alpha} disk.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.09370-b31b1b.svg)](https://arxiv.org/abs/2611.09370) | **The OpenMULTI3D stellar analysis package. A 3D non-LTE spectrum synthesis code for AFGKM-type stars within the DISPATCH framework**  |
|| <mark>R. Hoppe</mark>, et al. -- incl., <mark>M. Bergemann</mark>, <mark>P. Eitner</mark>, <mark>N. Storm</mark> |
|*Appeared on*| *2026-11-09*|
|*Comments*| *Accepted for publication in A&A. 22 pages, 12 figures, 9 tables. Code available at this https URL*|
|**Abstract**|            Upcoming large spectroscopic surveys such as WEAVE, 4MOST, and SDSS-V will provide millions of high-resolution stellar spectra. This motivates the development of efficient codes to perform three-dimensional (3D) non-local thermodynamic equilibrium (non-LTE) radiative transfer (RT) calculations, in order to meet the requirements on efficiency and data analysis quality set by these surveys. We present a new open access 3D non-LTE spectrum synthesis code OpenMULTI3D, based on its predecessor MULTI3D, which was never released. This code can be used to compute large grids of spectra of AFGKM-type stars and will aid the exploitation of large-scale spectroscopic surveys such as 4MOST. We summarise the main input micro- and macro-physics of the code, describe its numerical implementation and provide selected test cases. The outputs of the code are compared with standard results in the literature, including the NLTE departure coefficients in 1D and 3D, the 3D NLTE abundance corrections, 3D synthetic spectral energy distributions for three types of stars, and 3D NLTE Balmer-line profiles of a metal-poor red giant. We also demonstrate how the code can be used to compute time-dependent spectra of stars, which may be relevant in precision stellar physics and analyses of exoplanet spectra. OpenMULTI3D offers an accessible and efficient platform for high-fidelity stellar analysis in the era of large-scale spectroscopic surveys.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.09875-b31b1b.svg)](https://arxiv.org/abs/2611.09875) | **SN 2026ahvu in NGC 1433 May Be a Prompt Type Ia Supernova with a Delay Time Under 100 Myr**  |
|| S. K. Sarbadhicary, et al. -- incl., <mark>J. G. Lobos</mark>, <mark>E. Schinnerer</mark> |
|*Appeared on*| *2026-11-09*|
|*Comments*| *10 pages, 2 figures, 1 table. Submitted to ApJL*|
|**Abstract**|            SN 2026ahvu is a Type Ia supernova (SN) that occurred in the nearby galaxy NGC 1433. Using archival Hubble and JWST imaging from the PHANGS survey, we show that the SN lies within 0.19$''$ and 0.26$''$ of two UV-bright star clusters. This $0.19/0.26''$ separation corresponds to $8/11$ pc ($17/24$ pc) at the reported distances of $9.1$ and $18.9$ Mpc, respectively. The probability of finding an unrelated Type Ia SN this close to a cataloged star cluster by chance is only $\sim$0.4 %. Thus, SN 2026ahvu is likely the first Type Ia SN that can be associated with relatively young star clusters in a nearby galaxy. From spectral energy distribution fitting using HST+JWST (or HST-only) data, we find that the clusters have age estimates of $37_{-5}^{+3}$ ($57_{-9}^{+14}$) Myr and $95_{-13}^{+15}$ ($88_{-12}^{+14}$) Myr. Association with either cluster would mean that SN 2026ahvu should be considered a prompt SN Ia and provide one of the most concrete measurements to date of a prompt Ia delay time. The median age of the younger star cluster is close to the minimum possible delay time for SN Ia progenitors. If the SN is indeed associated with the younger cluster, it would imply that Type Ia SNe can occur almost immediately after white dwarf formation. If the SN is associated with the older cluster, this discovery still presents the first example of a prompt Type Ia SN with delay time under 100 Myr.         |

## Failed papers


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.08676-b31b1b.svg)](https://arxiv.org/abs/2611.08676) | **Expanding the Volatile Abundance Inventory for High-Priority K-Dwarf JWST Exoplanet Systems**  |
|| D. R. Coria, et al. -- incl., <mark>I. J. M. Crossfield</mark> |
|*Appeared on*| *2026-11-09*|
|*Comments*| *32 pages, 6 figures, 5 tables. Accepted for publication in ApJ (11/3/2026)*|
|**Abstract**|            There are exceedingly few chemical abundance measurements available in the literature for K dwarf stars, especially for those with T$_{eff} <$ 4700 K due to an incompatibility of traditional analyses with their molecular-line-ridden optical spectra. To address the need for K dwarf abundance constraints, we present an elemental ([C/H], [O/H]) and isotopic ($^{12}$C/$^{13}$C, $^{16}$O/$^{18}$O) abundance analysis for six K dwarf exoplanet host stars: K2-141, NGTS-10, WASP-43, WASP-69, WASP-80, and HATS-72. Each system hosts an exoplanet targeted by JWST characterization efforts, making host star abundances especially critical as baseline references. Using MARCS model atmospheres, the TurboSpectrum spectral synthesis code, and near-infrared IGRINS spectra, we derive novel [C/H] and [O/H] abundance ratios for these high-priority K dwarf host stars. We further investigate prospects for determining $^{12}$C/$^{13}$C and $^{16}$O/$^{18}$O ratios using $^{13}$C$^{16}$O and $^{12}$C$^{18}$O lines from the 2.3 micron CO bandhead. The observed spectra are too noisy to measure $^{16}$O/$^{18}$O ratios for the sample, but we do find sub-solar $^{12}$C/$^{13}$C ratios for five out of six targets. We also derive uniform gyrochronological ages for each host star to compare with galactic chemical evolution models and identify underlying abundance-age trends. NIR H+K band spectroscopy provides an alternative to optical, atomic line-derived carbon and oxygen abundances for cool dwarf stars. The growing catalog of precise host star elemental and isotopic abundances complements ongoing observations of their companions and provides a baseline for understanding their chemical composition, origin, and evolution.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.08824-b31b1b.svg)](https://arxiv.org/abs/2611.08824) | **Interferometric survey of stellar parameters. Reliable methodology for a direct measurement of the angular diameter in exoplanet host stars**  |
|| H. Nowacki, et al. -- incl., <mark>S. Kraus</mark> |
|*Appeared on*| *2026-11-09*|
|*Comments*| **|
|**Abstract**|            Since their pioneer observations more than thirty years ago, exoplanets have been a major topic among the astrophysics community. Despite their ubiquity, our knowledge of planetary parameters and internal compositions still mostly relies on stellar quantities estimated through evolutionary models, which suffer from a calibration bias, preventing the community to have accurate discussions on the diversity of planets across the Universe. We use our sample to demonstrate the current best reachable precision of the CHARA Array to be $\sim$ 2% on radius determination, with an error budget dominated by the uncertainty on the angular diameter, while systematics are shown to be of the order of 1% thanks to our methodology. While the precision matches values found in the literature, we demonstrate the uniqueness of the present study in the relevance of the results put forward. Finally, we show that our methods naturally translate into an improved accuracy of the planetary parameters, along with an improvement in precision by $\sim$ 50% in radius and $\sim$ 30% in mass.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.08953-b31b1b.svg)](https://arxiv.org/abs/2611.08953) | **System noise temperature measurements of a 19-element phased array feed**  |
|| S. Yu, et al. -- incl., <mark>J. Li</mark> |
|*Appeared on*| *2026-11-09*|
|*Comments*| *Accepted for publication in RAA*|
|**Abstract**|            We present multi-epoch system noise temperature measurements of a 19 element hexagonal room-temperature phased array feed (PAF) using a modified Y-factor method. A Python-based pipeline with iterative channel calibration and maxSNR beam forming is developed to do data reduction, and a covariance-based decomposition scheme is proposed to extract mutual coupling noise temperature Tmc. Long-term observations from March to June 2026 capture temporal receiver noise drift mainly induced by LNA temperature variations. Within the 1.37-1.47 GHz operating band, the measured beam equivalent system noise temperature of the PAF is about 73-89 K. The measured Tmc baseline ranges from about 2 K to 8 K, while under thermal loading, the resonant peak of Tmc can increase to more than 20 K. The baseline of Tmc is consistent with the S-parameter mea surements, which show the Tmc of the full array ranges less than 10.7 K. The baseline fluctuation of Tmc most likely arises from the superposition of thermal and non-thermal components of the spatial electromagnetic fields. A detachable stainless steel shield is deployed to quantify sidelobe spillover noise, and comparative tests show unshielded en vironmental interference increases receiver noise temperature by over 60 K. The proposed measurements and data processing framework provides a practical reference for noise per formance characterization of ground-based radio telescope PAF prototypes.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.09332-b31b1b.svg)](https://arxiv.org/abs/2611.09332) | **Successive Decimetric Bursts Associated with Breakout Magnetic Reconnection during the Decay Phase of an M-Class Flare**  |
|| S. Ma, et al. -- incl., <mark>M. Zhang</mark> |
|*Appeared on*| *2026-11-09*|
|*Comments*| *10 Figures*|
|**Abstract**|            Radio imaging spectroscopy diagnoses coronal electron acceleration and propagation by tracking energetic electron beams and locating energy-release sites. Using MUSER (0.4-2 GHz) and SDO data, we investigate decimetric radio emission during an eruption with an M1.1 flare on 17 December 2014. The eruption involves a twisted coronal structure evolving through three stages: a violent internal eruption, a rise-and-expansion phase, and a breakout eruption. The flare peaks at 01:50 UT during the internal eruption, while the cluster of successive type III-like decimetric bursts occurs in the decay phase, linked to the breakout eruption. The radio source coincides with a high-squashing-factor Q region from nonlinear force-free field (NLFFF) extrapolations, indicating a quasi-separatrix layer (QSL). We interpret the radio emission as consistent with plasma emission driven by nonthermal electrons accelerated by breakout reconnection at this QSL, between the outer envelope of the erupting structure and the surrounding corona. The successive bursts occur in the decay phase because breakout reconnection involves a larger, more energetic twisted structure at higher altitude, where an extended high-Q region, favorable acceleration geometry, and repeated sub-eruptions sustain efficient electron acceleration. Missing soft X-ray enhancement suggests most accelerated electrons escape upward or lose energy before reaching the chromosphere. The concentrated source positions of the successive radio bursts suggest pseudostreamers may act as particle accelerators, trapping electrons and directing them along a narrow layer. These results establish decimetric radio imaging as a powerful diagnostic of breakout reconnection and particle acceleration in the solar corona, with implications for energy release in pseudostreamer environments.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.09743-b31b1b.svg)](https://arxiv.org/abs/2611.09743) | **Does DESI point towards transient cosmic acceleration?**  |
|| M. Ahmad, et al. -- incl., <mark>M. Zhang</mark> |
|*Appeared on*| *2026-11-09*|
|*Comments*| *11 pages, 9 figures, all comments are welcome*|
|**Abstract**|            We investigate an oscillatory equation of state for dark energy that oscillates around the mean value $-1$ and includes a source term proportional to the cube of the number of e-folds. Due to this source term, the oscillation amplitude shifts towards the $w_{\rm de}<-1$ regime at higher redshifts, whereas in the future, the source term drives the oscillation amplitude towards the $w_{\rm de}>-1$ regime. As the equation of state enters the $w_{\rm de}>-1$ regime in the future, the dark energy density dilutes, causing the Hubble parameter to decrease asymptotically towards zero, so that the present acceleration is transient: the Universe transitions to decelerated expansion and approaches a static state as $a\to\infty$, without ever contracting. We compare this equation of state with BellDE, CPL and flat $\Lambda$CDM. We confront the model with a combination of datasets, including Planck, DESI BAO and three different supernova compilations: Pantheon+, Union3 and DES-Dovekie. We compare the log-evidence of each model with that of $\Lambda$CDM. The dynamical dark energy models yield $H_0$ values in the range $66.0^{+0.80}_{-0.79}$--$67.84^{+0.69}_{-0.74}\,{\rm km\,s^{-1}\,Mpc^{-1}}$, with matter density values in the range $\Omega_{\rm m}=0.3067^{+0.0073}_{-0.0072}$--$0.3271^{+0.0083}_{-0.0080}$ across all the combinations. These values are consistent with those of the baseline model, apart from a mild upward shift in $\Omega_{\rm m}$ for the Union3 combination. Among all the models, BellDE yields comparatively stronger Bayesian evidence than the other models, with weak evidence for the Pantheon+ compilation and moderate evidence for the Union3 and DES-Dovekie compilations. The sourced oscillatory model yields inconclusive evidence for Pantheon + and Union3, while weak evidence for DES-Dovekie relative to $\Lambda$CDM.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |

## Export documents

We now write the .md files and export relevant images

In [7]:
def export_markdown_summary(md: str, md_fname:str, directory: str):
    """Export MD document and associated relevant images"""
    import os
    import shutil
    import re

    if (os.path.exists(directory) and not os.path.isdir(directory)):
        raise RuntimeError(f"a non-directory file exists with name {directory:s}")

    if (not os.path.exists(directory)):
        print(f"creating directory {directory:s}")
        os.mkdir(directory)

    fig_fnames = (re.compile(r'\[Fig.*\]\((.*)\)').findall(md) + 
                  re.compile(r'\<img src="([^>\s]*)"[^>]*/>').findall(md))
    print("found figures", fig_fnames)
    for fname in fig_fnames:
        if 'http' in fname:
            # No need to copy online figures
            continue
        if not os.path.exists(fname):
            print("file not found", fname)
            continue
        print("copying ", fname, "to", directory)
        destdir = os.path.join(directory, os.path.dirname(fname))
        destfname = os.path.join(destdir, os.path.basename(fname))
        try:
            os.makedirs(destdir)
        except FileExistsError:
            pass
        shutil.copy(fname, destfname)
    with open(os.path.join(directory, md_fname), 'w') as fout:
        fout.write(md)
    print("exported in ", os.path.join(directory, md_fname))
    [print("    + " + os.path.join(directory,fk)) for fk in fig_fnames]

In [8]:
for paper_id, md in documents:
    export_markdown_summary(md, f"{paper_id:s}.md", '_build/html/')

found figures ['tmp_2611.08612/./figures/mass_growth_curves.png', 'tmp_2611.08612/./figures/Mdyn_Mstar_overview.png', 'tmp_2611.08612/./figures/sample_overview.png']
copying  tmp_2611.08612/./figures/mass_growth_curves.png to _build/html/
copying  tmp_2611.08612/./figures/Mdyn_Mstar_overview.png to _build/html/
copying  tmp_2611.08612/./figures/sample_overview.png to _build/html/
exported in  _build/html/2611.08612.md
    + _build/html/tmp_2611.08612/./figures/mass_growth_curves.png
    + _build/html/tmp_2611.08612/./figures/Mdyn_Mstar_overview.png
    + _build/html/tmp_2611.08612/./figures/sample_overview.png
found figures ['tmp_2611.08983/./feh_lz_distribution_iso.png', 'tmp_2611.08983/./lz_feh_pattern_with_family_errors.png', 'tmp_2611.08983/./lz_distribution.png']
copying  tmp_2611.08983/./feh_lz_distribution_iso.png to _build/html/
copying  tmp_2611.08983/./lz_feh_pattern_with_family_errors.png to _build/html/
copying  tmp_2611.08983/./lz_distribution.png to _build/html/
exported 

## Display the papers

Not necessary but allows for a quick check.

In [9]:
[display(Markdown(k[1])) for k in documents];

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\red}{\textcolor{red}}$
$\newcommand{\dd}{\ensuremath{\mathrm{d}}}$
$\newcommand{\msafit}{{\sc msafit}}$
$\newcommand{\ppxf}{{\sc ppxf}}$
$\newcommand{\vobsi}{V_{i,{\rm obs}}}$
$\newcommand{\sigmaobsi}{\sigma_{v,i,{\rm obs}}}$
$\newcommand{\kms}{km~s^{-1}}$
$\newcommand{\vsigma}{(V/\sigma)_{r_e}}$
$\newcommand{\arraystretch}{1.1}$
$\newcommand{\arraystretch}{1.1}$
$\newcommand{\arraystretch}{1.35}$</div>



<div id="title">

# Jeans Dynamical Constraints on the IMF in Massive Quiescent Galaxies at $z\sim0.7$ from JWST/NIRSpec

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.08612-b31b1b.svg)](https://arxiv.org/abs/2611.08612)<mark>Appeared on: 2026-11-09</mark> -  _Resubmitted to A&A after minor revisions; 15 pages, 6 figures (excluding appendices)_

</div>
<div id="authors">

M. Slob, et al. -- incl., <mark>A. d. Graaff</mark>

</div>
<div id="abstract">

**Abstract:** We present dynamical masses for nine massive quiescent galaxies at $z\sim0.7$ from the JWST-IMFERNO program, measured from ultra-deep ( $\sim32$ hr), spatially resolved NIRSpec/MSA spectroscopy combined with NIRCam imaging. The most massive of these galaxies have previously been found to have a bottom-heavy initial mass function (IMF), based on measurements of dwarf-sensitive absorption features. Here, we examine whether the stellar masses inferred for these IMFs are consistent with the dynamical masses of these galaxies. We derive the dynamical masses by including Jeans Anisotropic Modelling within a forward-modelling framework that accounts for instrumental optics, source morphology, and geometric effects. We compare the resulting dynamical masses to the stellar masses for different IMF assumptions. A Milky Way IMF produces physically plausible dynamical-to-stellar mass ratios but implies central dark matter fractions ( $f_{\rm DM,1R_e}=0.54\pm0.06$ ) that are a factor $\sim2$ higher compared to local ETGs and simulations. When uniformly applying the IMF that was measured from absorption features, the mean dark matter fraction becomes low ( $f_{\rm DM,1R_e}=0.02\pm0.18$ ), and the stellar masses exceed the dynamical mass at $2\sigma$ significance for two out of seven galaxies in the sample. When we adopt the measured IMF only in the central $R_e$ and use a MW IMF in the outskirts, we find realistic dark matter fractions ( $f_{\rm DM,1R_e}=0.28\pm0.12$ ) and consistent dynamical masses. These results support a scenario where the cores of the most massive quiescent galaxies at $z\sim0.7$ had a bottom-heavy IMF, similar to those observed in the centres of local early-type galaxies. Our results are consistent with an inside-out growth scenario, in which size and mass growth of massive galaxies proceeds after quenching through minor mergers, leaving the bottom-heavy cores mostly intact.

</div>

<div id="div_fig1">

<img src="tmp_2611.08612/./figures/mass_growth_curves.png" alt="Fig9" width="100%"/>

**Figure 9. -** Enclosed total mass as a function of intrinsic radius for the best-fit JAM models of the galaxies in our sample. The red dashed line indicates $1R_e$, the limit used to calculate the JAM masses in this work. The green dashed line indicates the stellar contribution to the mass, and the yellow dashed line indicates the DM contribution. The grey data-points show the enclosed masses from JAM models derived from the VLT/VIMOS LEGA-C spectra \citep{JvanHoudt2021}, where available. (*fig:mass_growth_curve*)

</div>
<div id="div_fig2">

<img src="tmp_2611.08612/./figures/Mdyn_Mstar_overview.png" alt="Fig10" width="100%"/>

**Figure 10. -** vJAM dynamical masses versus stellar masses, calculated using a \citet{PKroupa2001} IMF (left panel), the best-fit IMF from \citet{CCheng2026}(middle panel), and a radially varying IMF (right panel). The IMFERNO galaxies from our sample are shown as the red data points. The contours show the distribution of all quiescent galaxies in the LEGA-C survey, corrected for the relevant IMF using a relation between IMF mismatch parameter and velocity dispersion from \citet{TTreu2010}. We also show the running mean of the LEGA-C population as the dotted white line. The dashed line indicates a one-to-one relation, where points below this line fall in the unphysical regime.  (*fig:mjam_mstar*)

</div>
<div id="div_fig3">

<img src="tmp_2611.08612/./figures/sample_overview.png" alt="Fig5" width="100%"/>

**Figure 5. -** Top panels: 8"$\times$8" NIRCam RGB cutouts from COSMOS-Web imaging \citep{CCasey2023} for the galaxies in our sample, ordered by size. The colours of the panels correspond to the points in the bottom panels. Bottom left panel: stellar mass vs. size in NIRCam F150W for the galaxies in our sample (coloured points). We also show mass vs. HST F160W sizes for the population of LEGA-C galaxies from which our sample was selected (black stars). Bottom centre panel: rest-frame $U-V$ vs. $V-J$ colours for our sample (coloured points), and the general population of LEGA-C galaxies (black stars). We indicate the quiescent galaxy selection box, defined by \citet{AMuzzin2013b}. Bottom right panel: Location in the sky of the galaxies in our sample, with the NIRSpec/MSA footprint shown in black. We show all galaxies in COSMOS2025 with $0.71<z_{\rm phot}<0.74$ as gray points, with the corresponding density contours in the background. The relative sizes of the COSMOS2025 markers are proportional to their stellar mass. (*fig:sample_overview*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.08612"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\vdag}{(v)^\dagger}$
$\newcommand\aastex{AAS\TeX}$
$\newcommand\latex{La\TeX}$</div>



<div id="title">

# Identifying Chemically Coherent Stellar Families in the Galactic High-$\alpha$ Disk with Low-Resolution Spectroscopy

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.08983-b31b1b.svg)](https://arxiv.org/abs/2611.08983)<mark>Appeared on: 2026-11-09</mark> -  _21 pages, 14 figures, accepted to ApJ_

</div>
<div id="authors">

T. Wu, et al. -- incl., <mark>M. Zhang</mark>

</div>
<div id="abstract">

**Abstract:** Identification of chemical substructures in the Galactic high- $\alpha$ disk provides valuable insight into its formation history. In this work, we demonstrate the capability of low-resolution spectroscopy for this purpose by applying DBSCAN clustering to 87,275 red giants in the high- $\alpha$ disk using 18 elemental abundances derived from LAMOST low-resolution spectra ( $R\lesssim1800$ ). We assess the effectiveness of this approach by comparing the clustering results from the low-resolution data with those obtained by applying the same procedure to high-resolution survey abundances and to mock catalogs. These comparisons suggest that, despite the relatively large uncertainties in individual elemental abundances derived from low-resolution spectra, the astrophysical coherence among different elements provides sufficient information to identify chemically coherent families that may share similar enrichment histories without necessarily sharing a common birth environment. After scanning the parameter space of the DBSCAN procedure and merging the results from different parameter combinations, we identify 133 candidate chemical families containing 4,642 unique stars (5.3 \% of the high- $\alpha$ disk sample). These families exhibit diverse multi-element abundance patterns, with particularly prominent differences in neutron-capture elements at nearly identical [ Fe/H ] values. These families therefore offer new constraints on the diversity of chemical enrichment pathways and on the assembly history of the high- $\alpha$ disk.

</div>

<div id="div_fig1">

<img src="tmp_2611.08983/./feh_lz_distribution_iso.png" alt="Fig10" width="100%"/>

**Figure 10. -** The distributions of angular momentum $L_Z$(top panels) and metallicity [Fe/H](bottom panels) for the identified chemical families. The left panels show non-isolated families (with $W_{L_{z}}\geq500$$\mathrm{kpc km s^{-1}}$), while the right panels show isolated families (with $W_{L_{z}}<500$$\mathrm{kpc km s^{-1}}$). Colors indicate individual families, sorted from largest (red) to smallest (purple). The dashed black curves represent the stars not assigned to any cluster.  (*fig:lz_feh_kde*)

</div>
<div id="div_fig2">

<img src="tmp_2611.08983/./lz_feh_pattern_with_family_errors.png" alt="Fig11" width="100%"/>

**Figure 11. -** Median chemical abundance patterns of the identified families in three selected [Fe/H] bins. Each line represents the chemical pattern of each family and is color-coded by median angular momentum $L_Z$.
        The x-axis shows the 18 elemental abundances adopted in this work, including light elements to neutron-capture species. The black error bars represent the typical uncertainties of the family median abundances within each metallicity bin. The number of identified families in each metallicity bin is indicated in the top-left corner of each panel. (*fig:lzfehpatterns*)

</div>
<div id="div_fig3">

<img src="tmp_2611.08983/./lz_distribution.png" alt="Fig12" width="100%"/>

**Figure 12. -** Distribution of the angular momentum $L_z$ for the identified chemical families in the high-dimensional abundance space. Each colored curve represents the kernel density distribution of $L_z$ for an individual chemical family after the hierarchical merging procedure. Different panels display different subsets of chemical families for clarity. Many families exhibit relatively compact and distinct $L_z$ distributions, suggesting that chemically coherent structures identified in abundance space may also retain characteristic dynamical properties. Several families show broad or multi-peaked $L_z$ distributions, indicating possible dynamical mixing, radial migration, or the presence of chemically similar but dynamically complex populations. (*fig:lz_dis*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.08983"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\pltscale}{0.7}$
$\newcommand{\spc}[2]{#1 \textsc{\romannumeral #2}\xspace}$
$\newcommand{\Multi}{\texttt{MULTI}\xspace}$
$\newcommand{\MultiD}{\texttt{MULTI3D}\xspace}$
$\newcommand{\Dispatch}{\texttt{DISPATCH}\xspace}$
$\newcommand{\Must}{\texttt{M3DIS}\xspace}$
$\newcommand{\Mdis}{\texttt{M3DIS}\xspace}$
$\newcommand{\Turb}{\texttt{TurbospectrumNLTE}\xspace}$
$\newcommand{\Code}{\texttt{OpenMULTI3D}\xspace}$
$\newcommand{\Stagger}{\texttt{STAGGER}\xspace}$
$\newcommand{\Cobold}{\texttt{CO5BOLD}\xspace}$
$\newcommand{\Marcs}{\texttt{MARCS}\xspace}$
$\newcommand{\Muram}{\texttt{MURAM}\xspace}$
$\newcommand{\Bifrost}{\texttt{BIFROST}\xspace}$
$\newcommand{\UOP}{\texttt{UOP}\xspace}$
$\newcommand{\Balder}{\texttt{BALDER}\xspace}$
$\newcommand{\Teff}{T_{\text{eff}}\xspace}$
$\newcommand{\stellarparams}[3]{T_{\text{eff}}=\SI{#1}{K}, \log g=#2, \mathrm{[Fe/H]}=#3}$
$\newcommand{\rom}[1]{\MakeUppercase{\romannumeral #1}}$
$\newcommand{\arraystretch}{1.4}$</div>



<div id="title">

# The OpenMULTI3D stellar analysis package: A 3D non-LTE spectrum synthesis code for AFGKM-type stars   \ within the DISPATCH framework

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.09370-b31b1b.svg)](https://arxiv.org/abs/2611.09370)<mark>Appeared on: 2026-11-09</mark> -  _Accepted for publication in A&A. 22 pages, 12 figures, 9 tables. Code available at this https URL_

</div>
<div id="authors">

<mark>R. Hoppe</mark>, et al. -- incl., <mark>M. Bergemann</mark>, <mark>P. Eitner</mark>, <mark>N. Storm</mark>

</div>
<div id="abstract">

**Abstract:** Upcoming large spectroscopic surveys such as WEAVE, 4MOST, and SDSS-V will provide millions of high-resolution stellar spectra. This motivates the development of efficient codes to perform three-dimensional (3D) non-local thermodynamic equilibrium (non-LTE) radiative transfer (RT) calculations, in order to meet the requirements on efficiency and data analysis quality set by these surveys. We present a new open access 3D non-LTE spectrum synthesis code $\Code$ , based on its predecessor $\MultiD$ , which was never released. This code can be used to compute large grids of spectra of AFGKM-type stars and will aid the exploitation of large-scale spectroscopic surveys such as 4MOST. We summarise the main input micro- and macro-physics of the code, describe its numerical implementation and provide selected test cases. The outputs of the code are compared with standard results in the literature, including the NLTE departure coefficients in 1D and 3D, the 3D NLTE abundance corrections, 3D synthetic spectral energy distributions for three types of stars, and 3D NLTE Balmer-line profiles of a metal-poor red giant. We also demonstrate how the code can be used to compute time-dependent spectra of stars, which may be relevant in precision stellar physics and analyses of exoplanet spectra. $\Code$ offers an accessible and efficient platform for high-fidelity stellar analysis in the era of large-scale spectroscopic surveys.

</div>

<div id="div_fig1">

<img src="tmp_2611.09370/./Figures/time_series_rgb.png" alt="Fig8" width="100%"/>

**Figure 8. -** The left panel shows the time-variability of a 3D NLTE Mg line profile for a series of $\sim100$ snapshots, presented for a typical metal-poor red giant. The starting time is past the relaxation time ($\Teff$ convergence) of the atmospheric model. The intensity fluctuations are measured with respect to the time averaged mean. The first grey inset in the left panel shows the intensity fluctuation of the continuum only. The second grey inset shows the Doppler shift of the line core, which is also shown in wavelength space by the black solid line in the centre of the plot. The right panel shows the mean line profile and highlights the line profiles from two selected snapshots responsible for the minimum and maximum Doppler shifts. (*fig:time_rgb*)

</div>
<div id="div_fig2">

<img src="tmp_2611.09370/./Figures/3D_SED_comparison_MDW.png" alt="Fig1.1" width="33%"/><img src="tmp_2611.09370/./Figures/3D_SED_comparison_RGB.png" alt="Fig1.2" width="33%"/><img src="tmp_2611.09370/./Figures/3D_SED_comparison_RSG.png" alt="Fig1.3" width="33%"/>

**Figure 1. -** SED test calculation of a typical M dwarf, metal-poor red giant and red supergiant computed in 1D and 3D LTE using $\Code$. The spectra have been convolved to a resolution of $R = 100$ for easier visual comparison. The legend states the effective temperature of the spectra from direct integration. (*fig:sed_3d*)

</div>
<div id="div_fig3">

<img src="tmp_2611.09370/./Figures/balmer.png" alt="Fig7" width="100%"/>

**Figure 7. -** Synthetic spectra of a metal-poor red giant. The 3D spectra are computed from a single snapshot of an $\Mdis$ model atmosphere, and the 1D LTE spectra come from $\Marcs$ models. (a) The 3700--3900 Å region at the synthesis resolution, computed with the line lists from Table \ref{tab:line_lists} in 1D LTE (red) and 3D NLTE (black) where only H and Fe are treated in NLTE. The labels give the principal quantum numbers of the lower and upper levels of the Balmer lines. (b) The same spectra convolved to $R = 5000$, with 3D LTE added (blue). (c--e) H10, H$\gamma$ and H$\alpha$ at $R = 5000$, computed with hydrogen lines only. 3D NLTE (black) and 3D LTE (blue) are compared with 1D LTE on $\Marcs$ models of 5000, 5500 and 6000 K (red). (*fig:balmer*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.09370"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\vdag}{(v)^\dagger}$
$\newcommand\aastex{AAS\TeX}$
$\newcommand\latex{La\TeX}$
$\newcommand\Msun{M_{\odot}\xspace}$
$\newcommand\msunpc{M_{\odot} pc^{-2}}$
$\newcommand\atomh{\ion{H}{1}\xspace}$
$\newcommand\molh{H_2\xspace}$
$\newcommand\cmcube{cm^{-3}\xspace}$
$\newcommand\kms{km s^{-1}\xspace}$
$\newcommand\todo[1]{\textcolor{red}{#1}}$
$\newcommand\blank{\textcolor{purple}{XX}\xspace}$
$\newcommand\blankcite{\textcolor{orange}{[cite]}\xspace}$
$\newcommand\densitymodelone{\Sigma_g=10 \msunpc\xspace}$
$\newcommand\densitymodeltwo{\Sigma_g=30 \msunpc\xspace}$
$\newcommand{\sn}{SN 2026ahvu}$</div>



<div id="title">

# SN 2026ahvu in NGC 1433 May Be a Prompt Type Ia Supernova with a Delay Time Under 100 Myr

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.09875-b31b1b.svg)](https://arxiv.org/abs/2611.09875)<mark>Appeared on: 2026-11-09</mark> -  _10 pages, 2 figures, 1 table. Submitted to ApJL_

</div>
<div id="authors">

S. K. Sarbadhicary, et al. -- incl., <mark>J. G. Lobos</mark>, <mark>E. Schinnerer</mark>

</div>
<div id="abstract">

**Abstract:** $\sn$ is a Type Ia supernova (SN) that occurred in the nearby galaxy NGC 1433. Using archival Hubble and JWST imaging from the PHANGS survey, we show that the SN lies within 0.19 $"$ and 0.26 $"$ of two UV-bright star clusters. This $0.19/0.26"$ separation corresponds to $8/11$ pc ( $17/24$ pc) at the reported distances of $9.1$ and $18.9$ Mpc, respectively. The probability of finding an unrelated Type Ia SN this close to a cataloged star cluster by chance is only $\sim$ 0.4 \% . Thus, $\sn$ is likely the first Type Ia SN that can be associated with relatively young star clusters in a nearby galaxy. From spectral energy distribution fitting using HST+JWST (or HST-only) data, we find that the clusters have age estimates of $37_{-5}^{+3}$ ( $57_{-9}^{+14}$ ) Myr and $95_{-13}^{+15}$ ( $88_{-12}^{+14}$ ) Myr. Association with either cluster would mean that $\sn$ should be considered a prompt SN Ia and provide one of the most concrete measurements to date of a prompt Ia delay time. The median age of the younger star cluster is close to the minimum possible delay time for SN Ia progenitors. If the SN is indeed associated with the younger cluster, it would imply that Type Ia SNe can occur almost immediately after white dwarf formation. If the SN is associated with the older cluster, this discovery still presents the first example of a prompt Type Ia SN with delay time under 100 Myr.

</div>

<div id="div_fig1">

<img src="tmp_2611.09875/./fig1_alt_compressed.png" alt="Fig1" width="100%"/>

**Figure 1. -** **Location of $\sn$ in NGC 1433.**_Left:_ HST/WFC3 color image of the bar and nuclear ring of NGC 1433 (credit: ESA/Hubble \& NASA; F336W, F438W, F555W and F814W), rotated by 90$^\circ$. The black circle marks the TNS position of $\sn$(Type Ia). The grey circle marks SN 1985P (Type II), another historical SN in this galaxy within our 5$'$ footprint. The scale bar is $\approx 1$ kpc at $9.1$ Mpc or $\approx 2$ kpc ($22"$) at $d = 18.9$ Mpc. _Middle:_$2.5" \times 5"$ north-up color cutouts centered on $\sn$, from HST/UVIS (top; F275W+F336W, F438W, F555W, F814W) and JWST/NIRCam (bottom; F150W+F200W, F277W+F300M, F335M+F360M+F444W). The dashed white circle shows the $\sim$1$\sigma$($0.1"$) positional uncertainty of $\sn$. The cyan circles mark the two candidate host star clusters, $0.19"$ north (cluster 1) and $0.26"$ south (cluster 2) of the SN. The clusters are $0.45"$ apart, $\approx 20$ pc at 9.1 Mpc or 41 pc at 18.9 Mpc. _Right:_$1" \times 1"$ single-band cutouts on a common north-up grid, in order of wavelength: HST UV--optical, continuum-subtracted HST H$\alpha$, and JWST/NIRCam. Each panel uses an asinh stretch from $-1\sigma$ to its peak, clipped to 5--30$\sigma$. The markers are the same as in the middle panels, and the scale bar is $0.25"$(11 or 23 pc, respectively). Both clusters are detected from the UV to $\sim$4.4 $\mu$m. Neither cluster shows H$\alpha$ emission, and neither is detected by MIRI (not shown): the clusters lie in a gap between dust/PAH filaments. (*fig:fig1*)

</div>
<div id="div_fig2">

<img src="tmp_2611.09875/./ngc1433_E_BVmin_0p0_E_BVmax_30p0_Z0p02_v5p4_general_all_SN_bigap_1_kfhphot.png" alt="Fig2.1" width="50%"/><img src="tmp_2611.09875/./ngc1433_E_BVmin_0p0_E_BVmax_30p0_Z0p02_v5p4_general_all_SN_bigap_2_kfhphot.png" alt="Fig2.2" width="50%"/>

**Figure 2. -** Observed SED (open circles with error bars, see Table \ref{tab:photandfit}) with their best-fit modeled spectra (dashed dark red line) for Cluster 1 (top) and Cluster 2 (bottom). Solid dark red points show the modeled fluxes in each filter. The dotted line in Cluster 1 indicates the modeled intrinsic stellar continuum, without dust absorption or emission. Note the divergence of Cluster 1's NIR modeled spectrum from the pure stellar continuum, indicating that there must be some hot dust present. Best fit ages and masses are reported in Table \ref{tab:photandfit}, along with 16-84\% confidence ranges. The presence of bright near-UV emission in both clusters leads to fitted ages $\lesssim 100$ Myr and suggests that $\sn$ is a prompt SN Ia if it is associated with either cluster. (*fig:clustersed*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.09875"></div>

# Create HTML index

In [10]:
from datetime import datetime, timedelta, timezone
from glob import glob
import os

files = glob('_build/html/*.md')
days = 7
now = datetime.today()
res = []
for fk in files:
    stat_result = os.stat(fk).st_ctime
    modified = datetime.fromtimestamp(stat_result, tz=timezone.utc).replace(tzinfo=None)
    delta = now.today() - modified
    if delta <= timedelta(days=days):
        res.append((delta.seconds, fk))
res = [k[1] for k in reversed(sorted(res, key=lambda x:x[1]))]
npub = len(res)
print(len(res), f" publications files modified in the last {days:d} days.")
# [ print('\t', k) for k in res ];

139  publications files modified in the last 7 days.


In [11]:
import datetime
from glob import glob

def get_last_n_days(lst, days=1):
    """ Get the documents from the last n days """
    sorted_lst = sorted(lst, key=lambda x: x[1], reverse=True)
    for fname, date in sorted_lst:
        if date >= str(datetime.date.today() - datetime.timedelta(days=days)):
            yield fname

def extract_appearance_dates(lst_file):
    dates = []

    def get_date(line):
        return line\
            .split('Appeared on:')[-1]\
            .split('</mark>')[0].strip()

    for fname in lst:
        with open(fname, 'r') as f:
            found_date = False
            for line in f:
                if not found_date:
                    if "Appeared on" in line:
                        found_date = True
                        dates.append((fname, get_date(line)))
                else:
                    break
    return dates

from glob import glob
lst = glob('_build/html/*md')
days = 7
dates = extract_appearance_dates(lst)
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last {days:d} days.")

15  publications in the last 7 days.


In [12]:
def create_carousel(npub=4):
    """ Generate the HTML code for a carousel with `npub` slides """
    carousel = ["""  <div class="carousel" """,
                """       data-flickity='{ "autoPlay": 10000, "adaptiveHeight": true, "resize": true, "wrapAround": true, "pauseAutoPlayOnHover": true, "groupCells": 1 }' id="asyncTypeset">"""
                ]
    
    item_str = """    <div class="carousel-cell"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        carousel.append(item_str.format(k=k))
    carousel.append("  </div>")
    return '\n'.join(carousel)

def create_grid(npub=4):
    """ Generate the HTML code for a flat grid with `npub` slides """
    grid = ["""  <div class="grid"> """,
                ]
    
    item_str = """    <div class="grid-item"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        grid.append(item_str.format(k=k))
    grid.append("  </div>")
    return '\n'.join(grid)

In [13]:
carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "7-day archives" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
with open("_build/html/index_7days.html", 'w') as fout:
    fout.write(page)

In [14]:
# redo for today
days = 1
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last day.")

carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "Daily" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(carousel, docs, slides)
# print(page)
with open("_build/html/index_daily.html", 'w') as fout:
    fout.write(page)

4  publications in the last day.


In [15]:
# Create the flat grid of the last N papers (fixed number regardless of dates)
from itertools import islice 

npub = 6
res = [k[0] for k in (islice(reversed(sorted(dates, key=lambda x: x[1])), 6))]
print(len(res), f" {npub} publications selected.")

grid = create_grid(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("grid_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- grid-content:s --%}", grid)\
               .replace("{%-- suptitle:s --%}",  f"Last {npub:,d} papers" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(grid, docs, slides)
# print(page)
with open("_build/html/index_npub_grid.html", 'w') as fout:
    fout.write(page)

6  6 publications selected.
